## Open In Colab\n\n[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/paper_tables_figures_colab.ipynb)


# Risk/UQ Paper Track: Paper Tables and Figures

## Objective
Render publication-ready tables/figures from saved artifacts only.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 4 - Configuration + Run Context


In [ ]:
from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

# Mandatory contract fields
RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)

cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)
run_prefix = run_ctx.run_prefix

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_risk_uq_run_context(run_ctx, display_fn=display)


## Step 5 - Fast-Fail Validation (Contract Prereqs)


In [ ]:
from src.workflows import (
    load_notebook_contract_manifest,
    validate_notebook_contract_manifest,
    write_contract_storage_mirror,
    write_notebook_contract_manifest,
)

manifest = load_notebook_contract_manifest(cfg.run_prefix)
ok, reasons = validate_notebook_contract_manifest(
    manifest,
    require_quick_probe=True,
    require_preflight=True,
    required_stages=('risk_training_completed', 'uq_benchmark_completed'),
)
if not ok:
    raise RuntimeError(
        'Notebook contract prerequisites not met. Run training + benchmark notebooks first. '
        f'Reasons: {reasons}'
    )

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='paper_tables_figures_colab',
    stage='paper_export_started',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
)
contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='paper_export_started',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
)
print('manifest_path =', manifest_path)
print('contract_run_dir =', contract_paths['contract_run_dir'])


## Step 6 - Main Execution (Export)


In [ ]:
from src.workflows import export_paper_tables_and_figures

paper_bundle = None
if not bool(RUN_ENABLED):
    print('[main] skipped: RUN_ENABLED=False')
else:
    EXPORT_DIR = f"{cfg.run_prefix}_paper_exports"
    paper_bundle = export_paper_tables_and_figures(
        run_prefix=cfg.run_prefix,
        output_dir=EXPORT_DIR,
    )
    paper_bundle.exported_paths


## Step 7 - Evaluation/Preview


In [ ]:
import pandas as pd
from pathlib import Path

if paper_bundle is None:
    print('[report] no run output because RUN_ENABLED=False')
else:
    export_dir = Path(paper_bundle.output_dir)
    calib = pd.read_csv(export_dir / 'calibration_table.csv')
    shift = pd.read_csv(export_dir / 'shift_robustness_table.csv')
    tradeoff = pd.read_csv(export_dir / 'controller_tradeoff_table.csv')
    display(calib.head(20))
    display(shift.head(20))
    display(tradeoff.head(20))


## Step 8 - Export + Manifest (Completed)


In [ ]:
if paper_bundle is None:
    stage_name = 'paper_export_skipped'
    artifact_paths = {}
    metrics_path = None
    extra = {'run_skipped': 1}
else:
    stage_name = 'paper_export_completed'
    artifact_paths = dict(paper_bundle.exported_paths)
    metrics_path = str(artifact_paths.get('calibration_table', '')) or None
    extra = {
        'export_dir': str(paper_bundle.output_dir),
        'n_exported_paths': int(len(dict(paper_bundle.exported_paths))),
    }

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='paper_tables_figures_colab',
    stage=stage_name,
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage=stage_name,
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)
print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
